# 04 - Feature selection

Objetivo: detectar redundancia y conservar variables numericas interpretables antes del clustering.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import polars as pl
sys.path.insert(0, str(Path.cwd().parent))
from analysis.pipeline import load_transactions, build_customer_features

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
features = build_customer_features(load_transactions(ROOT / "transacciones_sinteticas.csv"))
feature_columns = [c for c in features.columns if c != "codigo_cliente"]
matrix = features.select(feature_columns).to_numpy().astype(float)
matrix = np.nan_to_num(matrix, nan=np.nanmedian(matrix, axis=0), posinf=0.0, neginf=0.0)
correlation = np.corrcoef(matrix, rowvar=False)
pl.DataFrame(correlation.round(2), schema=feature_columns).head()

In [ ]:
threshold = 0.95
redundant = []
for i, left in enumerate(feature_columns):
    for j in range(i + 1, len(feature_columns)):
        right = feature_columns[j]
        value = correlation[i, j]
        if abs(value) >= threshold:
            redundant.append({"feature": left, "redundant_with": right, "correlation": value})
redundant_table = pl.DataFrame(redundant) if redundant else pl.DataFrame({"feature": [], "redundant_with": [], "correlation": []})
redundant_table

In [ ]:
drop = {row["redundant_with"] for row in redundant}
selected_columns = [c for c in feature_columns if c not in drop]
selection = pl.DataFrame({"feature": selected_columns, "selected": True})
output = ROOT / "results" / "metrics" / "selected_features.csv"
output.parent.mkdir(parents=True, exist_ok=True)
selection.write_csv(output)
print(f"Variables iniciales: {len(feature_columns)}; seleccionadas: {len(selected_columns)}")
selected_columns

## Decision
La seleccion automatica solo elimina redundancia extrema. La relevancia analitica y la interpretabilidad se revisan junto con las metricas de clustering.